# 03 — Baseline: Douglas-Peucker + Area Filter

Before training a model, we need a **baseline to beat**.

This notebook implements the classical generalization pipeline:
1. **Area filter** — remove small polygons below a size threshold
2. **Douglas-Peucker simplification** — reduce vertex count
3. Evaluate against true BRT using IoU, Hausdorff distance, and polygon count ratio

The resulting metrics are saved and reused in notebook 05 (evaluation).


## 0. CONFIG

In [ ]:
from pathlib import Path

CONFIG = {
    "output_root": Path("processed"),
    "crs":         "EPSG:28992",

    # ── Baseline parameters (edit these to tune the baseline) ──

    # Polygons smaller than this area (m²) are removed
    # Try: 10, 25, 50, 100
    "min_area_m2": 25.0,

    # Douglas-Peucker tolerance in metres
    # Larger = more simplification, fewer vertices
    # Try: 0.5, 1.0, 2.0, 5.0
    "dp_tolerance_m": 1.0,

    # Evaluate on this many tiles (set to None to use all test tiles)
    "eval_n_tiles": 30,
}

## 1. Imports

In [ ]:
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.errors import ShapelyDeprecationWarning
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning)

print("Imports OK")

## 2. Baseline generalization functions

In [ ]:
def apply_area_filter(gdf: gpd.GeoDataFrame, min_area: float) -> gpd.GeoDataFrame:
    """
    Remove polygons smaller than min_area square metres.
    """
    mask = gdf.geometry.area >= min_area
    filtered = gdf[mask].copy()
    return filtered


def apply_douglas_peucker(gdf: gpd.GeoDataFrame, tolerance: float) -> gpd.GeoDataFrame:
    """
    Simplify polygon geometries using Douglas-Peucker.
    tolerance is in the CRS units (metres for RD New).
    Drops any polygons that become degenerate after simplification.
    """
    gdf = gdf.copy()
    gdf["geometry"] = gdf.geometry.simplify(tolerance, preserve_topology=True)

    # Remove degenerate geometries
    gdf = gdf[gdf.geometry.is_valid]
    gdf = gdf[~gdf.geometry.is_empty]
    gdf = gdf[gdf.geometry.area > 0]
    return gdf


def generalize_baseline(bgt: gpd.GeoDataFrame, config: dict) -> gpd.GeoDataFrame:
    """
    Full baseline pipeline:
      1. Area filter
      2. Douglas-Peucker simplification
    Returns the generalized GeoDataFrame.
    """
    step1 = apply_area_filter(bgt, config["min_area_m2"])
    step2 = apply_douglas_peucker(step1, config["dp_tolerance_m"])
    return step2


print("Baseline functions defined")

## 3. Evaluation metrics

In [ ]:
def iou_polygon(geom_a, geom_b) -> float:
    """Intersection over Union between two Shapely geometries."""
    try:
        inter = geom_a.intersection(geom_b).area
        union = geom_a.union(geom_b).area
        return inter / union if union > 0 else 0.0
    except Exception:
        return 0.0


def mean_hausdorff(pred: gpd.GeoDataFrame,
                   target: gpd.GeoDataFrame) -> float:
    """
    Mean Hausdorff distance between matched polygon pairs.
    Matching is done by spatial join (nearest centroid).
    """
    if len(pred) == 0 or len(target) == 0:
        return float("nan")

    # Match each pred polygon to the nearest target polygon
    pred_centroids = pred.copy()
    pred_centroids["geometry"] = pred.geometry.centroid

    joined = gpd.sjoin_nearest(
        pred_centroids[["geometry"]].reset_index(),
        target[["geometry"]].reset_index(),
        how="left",
        lsuffix="pred",
        rsuffix="tgt",
    )

    distances = []
    for _, row in joined.iterrows():
        try:
            pred_geom = pred.geometry.iloc[int(row["index_pred"])]
            tgt_geom  = target.geometry.iloc[int(row["index_tgt"])]
            distances.append(pred_geom.hausdorff_distance(tgt_geom))
        except Exception:
            pass

    return float(np.mean(distances)) if distances else float("nan")


def evaluate_tile(pred: gpd.GeoDataFrame,
                  target: gpd.GeoDataFrame) -> dict:
    """
    Compute all metrics for one tile.
    Returns a dict of scalar metric values.
    """
    if len(pred) == 0 or len(target) == 0:
        return {"mean_iou": 0.0, "hausdorff": float("nan"),
                "count_ratio": 0.0, "vertex_reduction": float("nan")}

    # Mean IoU: dissolve both to a single geometry and compare
    pred_union   = pred.geometry.union_all()
    target_union = target.geometry.union_all()
    mean_iou     = iou_polygon(pred_union, target_union)

    # Hausdorff
    hausdorff = mean_hausdorff(pred, target)

    # Polygon count ratio (how close are we to BRT polygon count?)
    count_ratio = len(pred) / len(target) if len(target) > 0 else float("nan")

    # Vertex reduction ratio
    def total_vertices(gdf):
        return sum(
            len(g.exterior.coords) if g.geom_type == "Polygon"
            else sum(len(p.exterior.coords) for p in g.geoms)
            for g in gdf.geometry
        )

    verts_pred   = total_vertices(pred)
    verts_target = total_vertices(target)
    vertex_reduction = 1.0 - (verts_pred / (verts_target + 1e-9))

    return {
        "mean_iou":         round(mean_iou, 4),
        "hausdorff":        round(hausdorff, 2),
        "count_ratio":      round(count_ratio, 3),
        "vertex_reduction": round(vertex_reduction, 4),
    }


print("Metric functions defined")

## 4. Run baseline on test tiles

In [ ]:
# Load tile index
with open(CONFIG["output_root"] / "tile_index.json") as f:
    tile_index = json.load(f)

test_tiles = tile_index["test"]
if CONFIG["eval_n_tiles"] is not None:
    test_tiles = test_tiles[:CONFIG["eval_n_tiles"]]

print(f"Evaluating baseline on {len(test_tiles)} test tiles...")
print(f"  Area filter   : >= {CONFIG['min_area_m2']} m²")
print(f"  DP tolerance  : {CONFIG['dp_tolerance_m']} m")

results = []

for i, tile in enumerate(test_tiles):
    bgt = gpd.read_file(tile["bgt"])
    brt = gpd.read_file(tile["brt"])

    # Run baseline generalization
    pred = generalize_baseline(bgt, CONFIG)

    # Evaluate
    metrics = evaluate_tile(pred, brt)
    metrics["tile"] = tile["bgt"]
    metrics["city"] = tile["city"]
    metrics["n_bgt"] = len(bgt)
    metrics["n_pred"] = len(pred)
    metrics["n_brt"] = len(brt)
    results.append(metrics)

    if (i + 1) % 10 == 0:
        print(f"  Processed {i+1}/{len(test_tiles)} tiles")

results_df = pd.DataFrame(results)
print(f"\nDone. {len(results_df)} tiles evaluated.")

In [ ]:
# Summary statistics
print("─" * 50)
print("BASELINE RESULTS — Test set")
print("─" * 50)
summary = results_df[["mean_iou", "hausdorff", "count_ratio", "vertex_reduction"]].describe()
print(summary.round(4).to_string())

# Save results for comparison in notebook 05
results_df.to_csv(CONFIG["output_root"] / "baseline_results.csv", index=False)
print(f"\nResults saved to: {CONFIG['output_root'] / 'baseline_results.csv'}")

## 5. Visualize baseline on one tile

In [ ]:
# Pick the tile closest to median IoU for a representative example
median_iou = results_df["mean_iou"].median()
best_idx = (results_df["mean_iou"] - median_iou).abs().idxmin()
example = test_tiles[best_idx]

bgt_ex  = gpd.read_file(example["bgt"])
brt_ex  = gpd.read_file(example["brt"])
pred_ex = generalize_baseline(bgt_ex, CONFIG)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

bgt_ex.plot(ax=axes[0],  color="steelblue", edgecolor="white", linewidth=0.3)
axes[0].set_title(f"BGT input ({len(bgt_ex)} polygons)")
axes[0].set_axis_off()

pred_ex.plot(ax=axes[1], color="darkorange", edgecolor="white", linewidth=0.3)
axes[1].set_title(f"Baseline output ({len(pred_ex)} polygons)")
axes[1].set_axis_off()

brt_ex.plot(ax=axes[2],  color="tomato",    edgecolor="white", linewidth=0.3)
axes[2].set_title(f"BRT target ({len(brt_ex)} polygons)")
axes[2].set_axis_off()

m = results_df.loc[best_idx]
plt.suptitle(
    f"Baseline example — IoU: {m['mean_iou']:.3f} | Hausdorff: {m['hausdorff']:.1f}m | "
    f"Count ratio: {m['count_ratio']:.2f}",
    fontsize=12
)
plt.tight_layout()
plt.savefig(CONFIG["output_root"] / "baseline_example.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Metric distribution plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(results_df["mean_iou"].dropna(), bins=20, color="steelblue", edgecolor="white")
axes[0].set_title("IoU distribution")
axes[0].set_xlabel("IoU")
axes[0].axvline(results_df["mean_iou"].mean(), color="red", linestyle="--",
                label=f"Mean: {results_df['mean_iou'].mean():.3f}")
axes[0].legend()

axes[1].hist(results_df["hausdorff"].dropna(), bins=20, color="darkorange", edgecolor="white")
axes[1].set_title("Hausdorff distance (m)")
axes[1].set_xlabel("Metres")
axes[1].axvline(results_df["hausdorff"].mean(), color="red", linestyle="--",
                label=f"Mean: {results_df['hausdorff'].mean():.1f}m")
axes[1].legend()

axes[2].hist(results_df["count_ratio"].dropna(), bins=20, color="tomato", edgecolor="white")
axes[2].set_title("Count ratio (pred/target)")
axes[2].set_xlabel("Ratio (1.0 = perfect)")
axes[2].axvline(1.0, color="black", linestyle="--", linewidth=1)
axes[2].axvline(results_df["count_ratio"].mean(), color="red", linestyle="--",
                label=f"Mean: {results_df['count_ratio'].mean():.2f}")
axes[2].legend()

plt.suptitle("Baseline metric distributions — Test set", fontsize=13)
plt.tight_layout()
plt.savefig(CONFIG["output_root"] / "baseline_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nBaseline summary:")
print(f"  Mean IoU          : {results_df['mean_iou'].mean():.4f}")
print(f"  Mean Hausdorff    : {results_df['hausdorff'].mean():.2f} m")
print(f"  Mean count ratio  : {results_df['count_ratio'].mean():.3f}")
print(f"  Mean vert. reduction: {results_df['vertex_reduction'].mean():.3f}")
print("\nThese are the numbers your model needs to beat.")